# Obtaining Data from AWS for Classification Task

# Data Download for Classification Task

We need the data to complete the classification challenge task.  

We **could** download the files and upload them to the repo, but that approach is **not repeatable**.  

Instead, we create a script that downloads the data directly from its source. The resulting data files **are not committed**, so the process remains reproducible.  

The script below uses the credentials provided with the challenge to connect to the AWS S3 bucket via a `boto3` session, and downloads the files into the `data/` folder.  

If the folder doesn’t exist yet, the script will create it automatically.  

We will work through the analysis **cell by cell**, with comments explaining each step.

In [ ]:
# Imports 

import os
import json
import requests
import boto3

In [ ]:
# Specify the credentials location -- location taken from the Challenge Readme, from the penultimate paragraph of the 
# section: 0. Setup: Data.

creds_url = "https://cct-ds-code-challenge-input-data.s3.af-south-1.amazonaws.com/ds_code_challenge_creds.json"

In [ ]:
# Obtain credentials in session. 

creds_response = requests.get(creds_url)
creds = creds_response.json()

In [ ]:
# Inspect 

creds

In [ ]:
# Assign access and secret key objects. 

AWS_ACCESS_KEY = creds['s3']['access_key']
AWS_SECRET_KEY = creds['s3']['secret_key']

In [ ]:
# Create a boto3 session using Credentials and the Region Name given to us in the Challenge ReadMe. 

# Create Session
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name="af-south-1"
)

# Create S3 Client 
s3 = session.client("s3")

In [ ]:
# Create a data dictionary (if it exists this request is ignored)

# os.makedirs("~/data", exist_ok=True)
data_dir = "data"          # ./data
os.makedirs(data_dir, exist_ok=True)

In [ ]:
# Specify a the S3 bucket and file name of the data we want to obtain. 

bucket = "cct-ds-code-challenge-input-data"
file_name = "city-hex-polygons-8.geojson"

# specify local path
local_path = f"data/{file_name}"

# download the bucket
s3.download_file(bucket, file_name, local_path)

# confirm download. 
print(f"Downloaded to {local_path}")

In [ ]:
# bucket and file name of the data we want to obtain. 

bucket = "cct-ds-code-challenge-input-data"
file_name = "sr_hex.csv.gz"

# specify local path
local_path = f"data/{file_name}"

# download the bucket
s3.download_file(bucket, file_name, local_path)

# confirm download. 
print(f"Downloaded to {local_path}"

In [ ]:
import random
# S3 setup
bucket = "cct-ds-code-challenge-input-data"

# Prefixes for classes
prefixes = {"yes": "images/swimming-pool/yes", "no": "images/swimming-pool/no"}

# Local folders
local_root = "data/sample_images"
os.makedirs(local_root, exist_ok=True)

# Number of images to sample per class
n_samples = 100

for label, prefix in prefixes.items():
    local_folder = os.path.join(local_root, label)
    os.makedirs(local_folder, exist_ok=True)
    
    # List objects in S3
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    all_keys = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith(".tif")]
    
    # Sample
    sampled_keys = random.sample(all_keys, min(n_samples, len(all_keys)))
    
    # Download
    for key in sampled_keys:
        local_path = os.path.join(local_folder, os.path.basename(key))
        s3.download_file(bucket, key, local_path)
        print(f"Downloaded {key} -> {local_path}")